<center><img src="../../image/dice.png" alt="DICE" style="width: 300px;"/></center>

# 12.OpenCV 手眼九点标定工作台

@－－－－DICE－－－－<br>
@  文件名：13.OpenCV 手眼九点标定工作台.py <br>
@  版本：V2.0 <br>
@  说明：我们第一步需要开启摄像头作为显示
我们来学习一下摄像头的开启方式之一吧，一般来说，对于我们使用摄像头的时候，首先我们需要测试我们的摄像头是否正常工作，画面是否清晰，让我们一起来测试我们的摄像头吧！！！<br>

我们开始测试摄像头是否可用

### 调整机械臂到达Home 点，便于获取图像

In [1]:
import math
from arm5dof_uservo import Arm5DoFUServo
from config import *

### 机械臂创建

In [2]:
arm = Arm5DoFUServo()

### 到达Home点

In [3]:
# 回到Home点
arm.home()

### 回到拍照点

In [4]:
! python3 set_arm_pose_by_name.py --pose_name capture_image

I0423 14:19:16.389792 548238389264 set_arm_pose_by_name.py:16] 位姿名称: capture_image
I0423 14:19:16.390177 548238389264 set_arm_pose_by_name.py:18] 机械臂初始化
I0423 14:19:18.467776 548238389264 set_arm_pose_by_name.py:20] 控制机械臂运动到: capture_image


### 运行标定程序，该程序需要在MobaXterm 中运行
python3 annotation_img9points.py

## 计算相机坐标系到工作台坐标系的空间变换

In [5]:
import numpy as np
import cv2 
import pandas as pd
import yaml

# 设置Numpy的打印选项
# 精确位数3，不启用科学计数法
np.set_printoptions(precision=3, suppress=True)

In [6]:
# 载入九点标定法的配置文件
handeye_9points_config = None
with open("../2.机械臂实验/config/handeye_9points.yaml", 'r', encoding='utf-8') as f:
	handeye_9points_config = yaml.load(f.read(), Loader=yaml.SafeLoader)

In [7]:
# 新的相机内参矩阵(畸变去除后)
M_intrisic_new = np.loadtxt('vconfig/M_intrisic_new.txt', delimiter=',')
# 因为是按畸变去除后的图像来计算，因此畸变系数为零向量
distor_coeff = np.float64([0, 0, 0, 0, 0]).astype("float32")

In [8]:
# 载入九点在2D图像中的坐标
img_9points = np.loadtxt("vconfig/img9points.txt", delimiter=',')
print(f"9点在像素坐标系的坐标: \n{img_9points}")


9点在像素坐标系的坐标: 
[[194. 207.]
 [345. 207.]
 [503. 205.]
 [175. 303.]
 [346. 303.]
 [525. 303.]
 [147. 430.]
 [344. 431.]
 [559. 437.]]


In [9]:
# 9点在工作台坐标系的坐标
# ws_9points_df = pd.read_excel("config/九点坐标-工作台坐标系.xlsx")
# ws_9points = np.float64(ws_9points_df.to_numpy())[:, 1:]
w = handeye_9points_config["board_width"]
h = handeye_9points_config["board_height"]
# P0点的坐标
x0 = 0.5 * h
y0 = 0.5 * w
ws_9points = np.float64([
	[x0, y0, 0], 	#P0
	[x0, 0, 0], 	#P1
	[x0, -y0, 0], 	#P2
 	[0, y0, 0], 	#P3
	[0, 0, 0], 		#P4
	[0, -y0, 0], 	#P5
	[-x0, y0, 0], 	#P6
	[-x0, 0, 0], 	#P7
	[-x0, -y0, 0], 	#P8
])
print(f"9点在工作台坐标系的坐标: \n{ws_9points}")

9点在工作台坐标系的坐标: 
[[ 50.   66.5   0. ]
 [ 50.    0.    0. ]
 [ 50.  -66.5   0. ]
 [  0.   66.5   0. ]
 [  0.    0.    0. ]
 [  0.  -66.5   0. ]
 [-50.   66.5   0. ]
 [-50.    0.    0. ]
 [-50.  -66.5   0. ]]


In [10]:
# 用solvePnPRansac会更稳一些
ret, rvec, tvec, *_ = cv2.solvePnPRansac(ws_9points, img_9points, \
                    M_intrisic_new, distor_coeff,  flags=cv2.SOLVEPNP_ITERATIVE)

In [11]:
# 构造相机坐标系到工作台坐标系的变换矩阵
T_cam2ws = np.eye(4)
T_cam2ws[:3, :3] = cv2.Rodrigues(rvec)[0]
T_cam2ws[:3, 3] = tvec.reshape(-1)
print("T_cam2ws")
print(T_cam2ws)
np.savetxt("config/T_cam2ws.txt", T_cam2ws, fmt='%.3f', delimiter=",")

T_cam2ws
[[  0.067  -0.995  -0.077  24.65 ]
 [ -0.687   0.01   -0.726  28.197]
 [  0.723   0.102  -0.683 254.18 ]
 [  0.      0.      0.      1.   ]]
